In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
from pathlib import Path
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

from src.utils.paths import load_paths
from src.utils.logging import setup_logger

from src.features.extract import load_feature_config, extract_features_from_flows, feature_config_hash_text
from src.pipeline.feature_pipeline import FeaturePipeline
from src.pipeline.artifacts import default_feature_artifacts

paths = load_paths()
paths.ensure_dirs()
logger = setup_logger(level="INFO")

logger.info(f"Repo root: {paths.repo_root}")
logger.info(f"Processed dir: {paths.data_processed}")
logger.info(f"Configs dir: {paths.configs_dir}")
logger.info(f"Artifacts dir: {paths.artifacts_dir}")


2026-02-28 14:59:10 | INFO | ai-vpn-firewall | Repo root: C:\Users\scoti\PycharmProjects\ai-vpn-firewall
2026-02-28 14:59:10 | INFO | ai-vpn-firewall | Processed dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed
2026-02-28 14:59:10 | INFO | ai-vpn-firewall | Configs dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\configs
2026-02-28 14:59:10 | INFO | ai-vpn-firewall | Artifacts dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts


In [2]:
features_yaml = paths.configs_dir / "features.yaml"
cfg = load_feature_config(features_yaml)

logger.info("Loading flows...")
vnat_flows = pd.read_parquet(paths.data_processed / "vnat" / "flows.parquet")
iscx_flows = pd.read_parquet(paths.data_processed / "iscx" / "flows.parquet")

logger.info("Extracting features from flows (VNAT)...")
vnat_feats = extract_features_from_flows(vnat_flows, cfg)
vnat_feats["dataset"] = "vnat"

logger.info("Extracting features from flows (ISCX)...")
iscx_feats = extract_features_from_flows(iscx_flows, cfg)
iscx_feats["dataset"] = "iscx"

# Bring split from old files (your existing approach)
vnat_old = pd.read_parquet(paths.data_processed / "vnat" / "features_trainable.parquet")
split_map_vnat = dict(zip(vnat_old["capture_id"], vnat_old["split"]))
vnat_feats["split"] = vnat_feats["capture_id"].map(split_map_vnat).fillna("unknown")

iscx_old = pd.read_parquet(paths.data_processed / "iscx" / "features.parquet")
split_map_iscx = dict(zip(iscx_old["capture_id"], iscx_old["split"]))
iscx_feats["split"] = iscx_feats["capture_id"].map(split_map_iscx).fillna("unknown")

# Drop unknown + apply min packets filter (your approach)
vnat_feats = vnat_feats[vnat_feats["split"] != "unknown"].copy()
iscx_feats = iscx_feats[iscx_feats["split"] != "unknown"].copy()

if "q_min_packets_ok" in vnat_feats.columns:
    vnat_feats = vnat_feats[vnat_feats["q_min_packets_ok"] == 1.0].copy()
if "q_min_packets_ok" in iscx_feats.columns:
    iscx_feats = iscx_feats[iscx_feats["q_min_packets_ok"] == 1.0].copy()

df_all = pd.concat([vnat_feats, iscx_feats], ignore_index=True)

# Normalize ISCX split naming (your approach)
df_all["split"] = df_all["split"].replace({"iscx_train": "train", "iscx_val": "val", "iscx_test": "test"})

logger.info(f"Combined df_all: {df_all.shape}, columns={len(df_all.columns)}")
df_all["split"].value_counts()


2026-02-28 14:59:10 | INFO | ai-vpn-firewall | Loading flows...
2026-02-28 14:59:11 | INFO | ai-vpn-firewall | Extracting features from flows (VNAT)...
2026-02-28 14:59:38 | INFO | ai-vpn-firewall | Extracting features from flows (ISCX)...
2026-02-28 14:59:59 | INFO | ai-vpn-firewall | Combined df_all: (12989, 60), columns=60


split
train    11306
val       1132
test       551
Name: count, dtype: int64

In [3]:
logger.info("Rebalancing VNAT splits...")
vnat_mask = df_all["dataset"] == "vnat"
vnat_df = df_all[vnat_mask].copy()

gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss.split(vnat_df, groups=vnat_df["capture_id"]))
vnat_train = vnat_df.iloc[train_idx].copy()
vnat_temp  = vnat_df.iloc[temp_idx].copy()

gss_val_test = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val_test.split(vnat_temp, groups=vnat_temp["capture_id"]))
vnat_val  = vnat_temp.iloc[val_idx].copy()
vnat_test = vnat_temp.iloc[test_idx].copy()

vnat_train["split"] = "train"
vnat_val["split"]   = "val"
vnat_test["split"]  = "test"

vnat_balanced = pd.concat([vnat_train, vnat_val, vnat_test], ignore_index=True)
df_all = pd.concat([df_all[~vnat_mask], vnat_balanced], ignore_index=True)

# Enforce Voipbuster split (match your real IDs)
vb_train_cap = "vpn_vpn_voipbuster1a.pcap"
vb_test_cap  = "vpn_vpn_voipbuster1b.pcap"

df_all.loc[df_all["capture_id"] == vb_train_cap, "split"] = "train"
df_all.loc[df_all["capture_id"] == vb_test_cap,  "split"] = "test"

print("Final splits:")
print(df_all["split"].value_counts())


2026-02-28 15:00:00 | INFO | ai-vpn-firewall | Rebalancing VNAT splits...
Final splits:
split
train    9428
val      2423
test     1138
Name: count, dtype: int64


In [4]:
def _pass(msg): print(f"PASS: {msg}")
def _warn(msg): print(f"WARNING: {msg}")
def _fail(msg): print(f"FAIL: {msg}")

def assert_true(cond: bool, ok: str, bad: str, warn: bool=False):
    if cond:
        _pass(ok)
        return True
    if warn:
        _warn(bad)
        return False
    _fail(bad)
    return False


In [5]:
def check_raw_flows(vnat_flows: pd.DataFrame, iscx_flows: pd.DataFrame):
    print("\n=== A) Raw flows sanity ===")
    assert_true(len(vnat_flows) > 0, "VNAT flows non-empty", "VNAT flows are empty")
    assert_true(len(iscx_flows) > 0, "ISCX flows non-empty", "ISCX flows are empty")

    required = ["capture_id"]
    missing_vnat = [c for c in required if c not in vnat_flows.columns]
    missing_iscx = [c for c in required if c not in iscx_flows.columns]

    assert_true(len(missing_vnat) == 0, "VNAT required columns exist", f"VNAT missing columns: {missing_vnat}")
    assert_true(len(missing_iscx) == 0, "ISCX required columns exist", f"ISCX missing columns: {missing_iscx}")

    # A3) Label distribution at flow level (per dataset)
    print("\nLabel distribution (Flows):")
    if not vnat_flows.empty:
        print("   VNAT:")
        print(vnat_flows["label"].value_counts(normalize=True))
    if not iscx_flows.empty:
        print("   ISCX:")
        print(iscx_flows["label"].value_counts(normalize=True))

check_raw_flows(vnat_flows, iscx_flows)



=== A) Raw flows sanity ===
PASS: VNAT flows non-empty
PASS: ISCX flows non-empty
PASS: VNAT required columns exist
PASS: ISCX required columns exist

Label distribution (Flows):
   VNAT:
label
0    0.988757
1    0.011243
Name: proportion, dtype: float64
   ISCX:
label
1    0.649178
0    0.350822
Name: proportion, dtype: float64


In [6]:
def check_features(df_all: pd.DataFrame, paths):
    print("\n=== B) Feature correctness ===")

    # required metadata columns
    meta_required = ["capture_id", "dataset", "split", "label"]
    missing = [c for c in meta_required if c not in df_all.columns]
    assert_true(len(missing) == 0, "Metadata columns present", f"Missing metadata columns: {missing}")

    # feature columns = numeric columns excluding obvious meta
    meta_cols = {"flow_id","capture_id","dataset","split","label","q_min_packets_ok"}
    feature_cols = [c for c in df_all.columns if c not in meta_cols]

    # only keep numeric features for NaN/Inf checks
    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_all[c])]
    assert_true(len(num_cols) > 0, "Found numeric feature columns", "No numeric feature columns found")

    df_valid = df_all
    if "q_min_packets_ok" in df_all.columns:
        df_valid = df_all[df_all["q_min_packets_ok"] == 1.0].copy()

    n_nans = int(df_valid[num_cols].isna().sum().sum())
    n_infs = int(np.isinf(df_valid[num_cols].to_numpy()).sum())
    assert_true(n_nans == 0, "No NaNs in numeric features", f"Found {n_nans} NaNs in numeric features")
    assert_true(n_infs == 0, "No Infs in numeric features", f"Found {n_infs} Infs in numeric features")

    # schema match between datasets
    v_cols = set(df_all[df_all["dataset"]=="vnat"].columns)
    i_cols = set(df_all[df_all["dataset"]=="iscx"].columns)
    assert_true(v_cols == i_cols, "VNAT/ISCX columns match", f"Schema mismatch: VNAT-only={len(v_cols-i_cols)}, ISCX-only={len(i_cols-v_cols)}")

    # duplicates
    if "flow_id" in df_all.columns:
        dup = int(df_all.duplicated(subset=["flow_id"]).sum())
        assert_true(dup == 0, "flow_id is unique", f"Found {dup} duplicated flow_id values", warn=True)
    else:
        _warn("flow_id column not present, skipping duplicate check")

    # B4) Feature config hash + feature set consistency
    features_yaml = paths.configs_dir / "features.yaml"
    current_hash = feature_config_hash_text(features_yaml)
    _pass(f"Current feature config hash: {current_hash}")

    # B6) Feature range plausibility checks
    # Check IAT burstiness (should be >= 0)
    if "f_iat_burstiness" in df_valid.columns:
        min_burst = df_valid["f_iat_burstiness"].min()
        assert_true(min_burst >= 0, "IAT burstiness non-negative", f"Negative burstiness found: {min_burst}")

    # Check up/down byte ratios (0 to 1)
    if "f_up_byte_ratio" in df_valid.columns:
        r_min = df_valid["f_up_byte_ratio"].min()
        r_max = df_valid["f_up_byte_ratio"].max()
        assert_true(r_min >= 0 and r_max <= 1.0 + 1e-9, "Byte ratio in [0,1]", f"Byte ratio out of bounds: [{r_min}, {r_max}]")

check_features(df_all, paths)



=== B) Feature correctness ===
PASS: Metadata columns present
PASS: Found numeric feature columns
PASS: No NaNs in numeric features
PASS: No Infs in numeric features
PASS: VNAT/ISCX columns match
PASS: flow_id is unique
PASS: Current feature config hash: b34b16527d5a4daeda64ce98a591ab6c5486249b452d15839ce316600ade813e
PASS: IAT burstiness non-negative
PASS: Byte ratio in [0,1]


In [7]:
def check_splits(df_all: pd.DataFrame):
    print("\n=== C) Split integrity ===")
    expected = {"train","val","test"}
    splits = set(df_all["split"].unique())
    assert_true(splits.issubset(expected), "Only train/val/test split values", f"Unexpected split values: {splits-expected}")

    # capture_id leakage
    train_caps = set(df_all[df_all["split"]=="train"]["capture_id"].unique())
    val_caps   = set(df_all[df_all["split"]=="val"]["capture_id"].unique())
    test_caps  = set(df_all[df_all["split"]=="test"]["capture_id"].unique())

    assert_true(len(train_caps & val_caps) == 0, "No capture overlap train<->val", f"Leak train/val: {len(train_caps & val_caps)} captures")
    assert_true(len(train_caps & test_caps) == 0, "No capture overlap train<->test", f"Leak train/test: {len(train_caps & test_caps)} captures")
    assert_true(len(val_caps & test_caps) == 0, "No capture overlap val<->test", f"Leak val/test: {len(val_caps & test_caps)} captures")

    # dataset distribution table
    print("\nDataset x Split (counts):")
    print(df_all.groupby(["dataset","split"]).size().unstack(fill_value=0))

    # C12) Label balance per split AND per dataset
    print("\nLabel balance per split:")
    print(df_all.groupby("split")["label"].value_counts(normalize=True).unstack())

    print("\nLabel balance per dataset per split:")
    print(df_all.groupby(["dataset", "split"])["label"].value_counts(normalize=True).unstack())

    # voipbuster check
    vb1a = df_all[df_all["capture_id"]=="vpn_vpn_voipbuster1a.pcap"]
    vb1b = df_all[df_all["capture_id"]=="vpn_vpn_voipbuster1b.pcap"]
    if len(vb1a): print("\nVoipbuster1a splits:", vb1a["split"].unique())
    if len(vb1b): print("Voipbuster1b splits:", vb1b["split"].unique())

check_splits(df_all)



=== C) Split integrity ===
PASS: Only train/val/test split values
PASS: No capture overlap train<->val
PASS: No capture overlap train<->test
PASS: No capture overlap val<->test

Dataset x Split (counts):
split    test  train   val
dataset                   
iscx      337   3614   931
vnat      801   5814  1492

Label balance per split:
label         0         1
split                    
test   0.783831  0.216169
train  0.753394  0.246606
val    0.692117  0.307883

Label balance per dataset per split:
label                 0         1
dataset split                    
iscx    test   0.305638  0.694362
        train  0.422524  0.577476
        val    0.331901  0.668099
vnat    test   0.985019  0.014981
        train  0.959064  0.040936
        val    0.916890  0.083110

Voipbuster1a splits: <ArrowStringArray>
['train']
Length: 1, dtype: str
Voipbuster1b splits: <ArrowStringArray>
['test']
Length: 1, dtype: str


In [8]:
def check_pipeline(df_all: pd.DataFrame):
    print("\n=== D) Pipeline correctness ===")

    df_train = df_all[df_all["split"]=="train"].copy()
    df_val   = df_all[df_all["split"]=="val"].copy()
    df_test  = df_all[df_all["split"]=="test"].copy()

    assert_true(len(df_train) > 0, "Train split non-empty", "Train split is empty")
    assert_true(len(df_val) > 0, "Val split non-empty", "Val split is empty", warn=True)
    assert_true(len(df_test) > 0, "Test split non-empty", "Test split is empty", warn=True)

    pipe = FeaturePipeline().fit(df_train)
    feats = pipe.model_feature_names()
    _pass(f"Pipeline fitted on train only. Model features: {len(feats)}")

    # D14) Pipeline feature list stability verification
    assert_true(len(feats) > 10, "Pipeline produced > 10 features", f"Pipeline produced only {len(feats)} features")

    # Check if q_min_packets_ok is excluded
    if "q_min_packets_ok" in feats:
        _fail("q_min_packets_ok found in model features! It should be excluded.")
    else:
        _pass("q_min_packets_ok correctly excluded from model features.")

    X_train = pipe.transform(df_train)
    X_val   = pipe.transform(df_val) if len(df_val) else None
    X_test  = pipe.transform(df_test) if len(df_test) else None

    if X_val is not None:
        assert_true(X_train.shape[1] == X_val.shape[1], "Train/Val feature dims match", f"Mismatch: train={X_train.shape}, val={X_val.shape}")
    if X_test is not None:
        assert_true(X_train.shape[1] == X_test.shape[1], "Train/Test feature dims match", f"Mismatch: train={X_train.shape}, test={X_test.shape}")

    # NaNs after transform
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns

    n_nans_train = int(X_train[numeric_cols].isna().sum().sum())
    assert_true(n_nans_train == 0, "No NaNs in X_train", f"NaNs found in X_train: {n_nans_train}")

    if X_val is not None:
        n_nans_val = int(X_val[numeric_cols].isna().sum().sum())
        assert_true(n_nans_val == 0, "No NaNs in X_val", f"NaNs found in X_val: {n_nans_val}")

    if X_test is not None:
        n_nans_test = int(X_test[numeric_cols].isna().sum().sum())
        assert_true(n_nans_test == 0, "No NaNs in X_test", f"NaNs found in X_test: {n_nans_test}")

    # zero variance
    stds = X_train[numeric_cols].std(axis=0)
    zero_var = int((stds == 0).sum())
    if zero_var:
        _warn(f"{zero_var} zero-variance features in X_train (not always fatal, but watch it).")
    else:
        _pass("No zero-variance features in X_train.")

    return pipe

pipeline = check_pipeline(df_all)



=== D) Pipeline correctness ===
PASS: Train split non-empty
PASS: Val split non-empty
PASS: Test split non-empty
PASS: Pipeline fitted on train only. Model features: 54
PASS: Pipeline produced > 10 features
PASS: q_min_packets_ok correctly excluded from model features.
PASS: Train/Val feature dims match
PASS: Train/Test feature dims match
PASS: No NaNs in X_train
PASS: No NaNs in X_val
PASS: No NaNs in X_test
PASS: No zero-variance features in X_train.


In [9]:
def smoke_random_label(df_all: pd.DataFrame, pipeline: FeaturePipeline, seed=42):
    print("\n=== E1) Smoke test: random labels (Permuted) ===")
    df_train = df_all[df_all["split"]=="train"].copy()
    df_test  = df_all[df_all["split"]=="test"].copy()
    assert_true(len(df_train) > 0 and len(df_test) > 0, "Train and test available", "Missing train/test for smoke test")

    X_train = pipeline.transform(df_train)
    X_test  = pipeline.transform(df_test)

    feat_cols = pipeline.model_feature_names()

    # Permute labels to preserve class balance
    rng = np.random.default_rng(seed)
    y_train_shuf = rng.permutation(df_train["label"].to_numpy())
    y_test = df_test["label"].to_numpy()

    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.1,
        n_jobs=4,
        eval_metric="auc",
        random_state=seed
    )
    model.fit(X_train[feat_cols], y_train_shuf)

    p = model.predict_proba(X_test[feat_cols])[:, 1]
    auc = roc_auc_score(y_test, p)
    print(f"Random-label TEST AUC: {auc:.4f}")

    if auc > 0.60:
        _fail("AUC is too high for random labels -> likely leakage or an index/merge bug.")
    else:
        _pass("AUC looks near chance -> good sign (no obvious leakage).")

smoke_random_label(df_all, pipeline)



=== E1) Smoke test: random labels (Permuted) ===
PASS: Train and test available
Random-label TEST AUC: 0.5417
PASS: AUC looks near chance -> good sign (no obvious leakage).


In [10]:
def smoke_cross_domain(df_all: pd.DataFrame, paths):
    print("\n=== E2) Smoke test: Cross-domain (VNAT -> ISCX) ===")
    # E18) Train on VNAT only -> test on ISCX only

    df_vnat_train = df_all[(df_all["dataset"] == "vnat") & (df_all["split"] == "train")].copy()
    df_iscx_test = df_all[(df_all["dataset"] == "iscx") & (df_all["split"] == "test")].copy()

    if df_vnat_train.empty or df_iscx_test.empty:
        _warn("Skipping cross-domain test: missing VNAT train or ISCX test data.")
        return

    # Fit pipeline on VNAT only
    pipe_vnat = FeaturePipeline().fit(df_vnat_train)
    feat_cols = pipe_vnat.model_feature_names()

    X_vnat = pipe_vnat.transform(df_vnat_train)
    X_iscx = pipe_vnat.transform(df_iscx_test)

    # DEBUG: Dump shapes and labels for comparison with notebook 10
    print("DEBUG Cross-Domain (NB16):")
    print("VNAT train shape:", X_vnat[feat_cols].shape)
    print("ISCX test shape:", X_iscx[feat_cols].shape)
    print("ISCX test label counts:", np.bincount(df_iscx_test["label"].astype(int)))

    y_vnat = df_vnat_train["label"].values
    y_iscx = df_iscx_test["label"].values

    model = xgb.XGBClassifier(
        n_estimators=30,
        max_depth=3,
        learning_rate=0.1,
        n_jobs=4,
        eval_metric="auc",
        random_state=42
    )
    model.fit(X_vnat[feat_cols], y_vnat)

    preds = model.predict_proba(X_iscx[feat_cols])[:, 1]

    # Save for comparison
    np.save(paths.artifacts_dir / "debug_y_iscx_nb16.npy", y_iscx.astype(int))
    np.save(paths.artifacts_dir / "debug_p_iscx_nb16.npy", preds)

    auc = roc_auc_score(y_iscx, preds)
    print(f"VNAT->ISCX AUC: {auc:.4f}")

    if auc < 0.5:
        _warn("Cross-domain AUC < 0.5 (worse than random). Domain shift might be severe.")
    elif auc > 0.99:
        _warn("Cross-domain AUC > 0.99 (suspiciously perfect). Check for leakage.")
    else:
        _pass("Cross-domain AUC is in plausible range [0.5, 0.99].")

smoke_cross_domain(df_all, paths)



=== E2) Smoke test: Cross-domain (VNAT -> ISCX) ===
DEBUG Cross-Domain (NB16):
VNAT train shape: (5814, 54)
ISCX test shape: (337, 54)
ISCX test label counts: [103 234]
VNAT->ISCX AUC: 0.2207


In [11]:
def smoke_holdout_capture(df_all: pd.DataFrame):
    print("\n=== E3) Smoke test: Capture holdout (Corrected) ===")
    # E19) Capture-level holdout sanity

    # Pick a large capture from train to hold out
    train_caps = df_all[df_all["split"] == "train"]["capture_id"].unique()

    # Try to find a mixed-label capture
    holdout_cap = None
    for cap in train_caps:
        subset = df_all[df_all["capture_id"] == cap]
        if subset["label"].nunique() > 1:
            holdout_cap = cap
            break

    if holdout_cap is None and len(train_caps) > 0:
        holdout_cap = train_caps[0]
        _warn("No mixed-label capture found for holdout. Using single-class capture.")

    if holdout_cap is None:
        _warn("Not enough training captures to perform holdout test.")
        return

    print(f"Holding out capture: {holdout_cap}")

    # Create temporary split
    mask_holdout = df_all["capture_id"] == holdout_cap
    df_ho_train = df_all[(~mask_holdout) & (df_all["split"] == "train")]
    df_ho_test = df_all[mask_holdout]

    # CRITICAL: Fit NEW pipeline
    pipe_ho = FeaturePipeline().fit(df_ho_train)
    feat_cols = pipe_ho.model_feature_names()

    X_train = pipe_ho.transform(df_ho_train)
    X_test = pipe_ho.transform(df_ho_test)

    y_train = df_ho_train["label"].values
    y_test = df_ho_test["label"].values

    model = xgb.XGBClassifier(
        n_estimators=30,
        max_depth=3,
        learning_rate=0.1,
        n_jobs=4,
        eval_metric="auc",
        random_state=42
    )
    model.fit(X_train[feat_cols], y_train)

    preds = model.predict_proba(X_test[feat_cols])[:, 1]

    if len(np.unique(y_test)) > 1:
        auc = roc_auc_score(y_test, preds)
        print(f"Holdout AUC: {auc:.4f}")
        if auc < 0.5:
            _warn("Holdout AUC < 0.5. This capture might be an outlier.")
        else:
            _pass("Holdout AUC >= 0.5.")
    else:
        _warn("Holdout set has only one class. AUC undefined.")
        acc = ((preds >= 0.5) == y_test).mean()
        print(f"Holdout Accuracy: {acc:.4f}")
        print(f"Mean Predicted Prob: {preds.mean():.4f} (True Label: {y_test[0]})")

smoke_holdout_capture(df_all)



=== E3) Smoke test: Capture holdout (Corrected) ===
Holding out capture: nonvpn_aim_chat_3a.pcap
Holdout Accuracy: 0.8462
Mean Predicted Prob: 0.2195 (True Label: 0)
